In [ ]:
from dataclasses import dataclass
from typing import Literal
from math import prod

@dataclass
class Antenna:
    name: str
    power: float
    combinability: float = 0.75

relay_antennae = {
    "HG-5 High Gain Antenna": Antenna("HG-5 High Gain Antenna", 5e6, 0.75),
    "RA-2 Relay Antenna": Antenna("RA-2 Relay Antenna", 2e9, 0.75),
    "RA-15 Relay Antenna": Antenna("RA-15 Relay Antenna", 15e9, 0.75),
    "RA-100 Relay Antenna": Antenna("RA-100 Relay Antenna", 100e9, 0.75),
}

direct_antennae = {
    "Communotron 16": Antenna("Communotron 16", 500e3, 1.00),
    "Communotron 16-S": Antenna("Communotron 16-S", 500e3, 0.00),
    "Communotron DTS-M1": Antenna("Communotron DTS-M1", 2e9, 0.75),
    "Communotron HG-55": Antenna("Communotron HG-55", 15e9, 0.75),
    "Communotron 88-88": Antenna("Communotron 88-88", 100e9, 0.75),
}

tracking_stations = {
    "Level 1": Antenna("Level 1", 2e9),
    "Level 2": Antenna("Level 2", 50e9),
    "Level 3": Antenna("Level 3", 250e9),
}

distances_to_kerbin = {
    "Moho": (7289385437, 19913538689),
    "Eve": (3668828971, 23530851543),
    "Gilly": (3668828971, 23579676543),
    "Kerbin": (0, 0),
    "Mun": (12000000, 12000000),
    "Minmus": (47000000, 47000000),
    "Duna": (6069283350, 35383028257),
    "Ike": (6069283350, 35386324257),
    "Dres": (21402401940, 60320789167),
    "Jool": (51735042066, 85812077664),
    "Laythe": (51735042066, 85839261664),
    "Vall": (51735042066, 85855229664),
    "Tylo": (51735042066, 85880577664),
    "Bop": (51735042066, 85970775164),
    "Pol": (51735042066, 86022701871),
    "Eeloo": (53183306389, 127081753657),
    # "Sarnus": (105481041293, 146116001357),
    # "Hale": (105481041293, 146126489588),
    # "Ovok": (105481041293, 146128292464),
    # "Eeloo (OPM)": (105481041293, 146135172295),
    # "Slate": (105481041293, 146160298021),
    # "Tekto": (105481041293, 146216082610),
    # "Urlum": (229218312140, 279415712909),
    # "Polta": (229218312140, 279427458396),
    # "Priax": (229218312140, 279427458396),
    # "Wal": (229218312140, 279484820311),
    # "Tal": (229218312140, 279487929474),
    # "Neidon": (390533330076, 428177051986),
    # "Thatmo": (390533330076, 428209366770),
    # "Nissee": (390533330076, 429015970830),
    # "Plock": (382964603173, 688705350475),
    # "Karen": (382964603173, 688707808275),
}

class Vessel:
    def __init__(self, name):
        self.name = name
        self.antennae : list[Antenna] = []
        self.power = 0.0
    def add_antenna(self, antenna: Antenna):
        self.antennae.append(antenna)
        self.power = self.compute_power()
    def compute_power(self):
        # strongest antenna power * (sum antenna power / strongest antenna power) ^ average combinability exponent
        strongest = max(antenna.power for antenna in self.antennae)
        sum_antenna_power = sum(antenna.power for antenna in self.antennae)
        ace = sum(antenna.combinability*antenna.power for antenna in self.antennae) / sum_antenna_power
        return strongest * (sum_antenna_power / strongest) ** ace

def compute_max_range(vessel_1: Vessel, vessel_2: Vessel) -> float:
    # range = sqrt(power_1 * power_2)
    return (vessel_1.power * vessel_2.power) ** 0.5

def compute_single_hop_signal_strength(vessel_1: Vessel, vessel_2: Vessel, distance: float) -> float:
    # cubic function of distance/max_range, ranging from 0 to 1
    max_range = compute_max_range(vessel_1, vessel_2)
    x = 1 - distance/max_range
    signal_strength = -2*x**3 + 3*x**2
    return signal_strength

def compute_multi_hop_signal_strength(vessels: list[Vessel], distances: list[float]) -> float:
    """
    Compute the multi-hop signal strength as the product of the single-hop signal strengths

    Parameters:
    vessels: list of vessels in the communication path, including the source and destination
    distances: list of distances between each pair of vessels in the communication path, in the same order as the vessels list (length should be len(vessels)-1)

    Returns:
    The multi-hop signal strength, as a value between 0 and 1
    """
    signal_strengths = [compute_single_hop_signal_strength(vessels[i], vessels[i+1], distances[i]) for i in range(len(vessels)-1)]
    return prod(signal_strengths)

Tracking_Station_2 = Vessel("Tracking Station 2")
Tracking_Station_2.add_antenna(tracking_stations["Level 2"])

Eve_Probe = Vessel("Eve Probe")
Eve_Probe.add_antenna(direct_antennae["Communotron HG-55"])
Eve_Probe.add_antenna(direct_antennae["Communotron HG-55"])

compute_multi_hop_signal_strength([Eve_Probe, Tracking_Station_2], [distances_to_kerbin["Eve"][1]])
compute_multi_hop_signal_strength([Eve_Probe, Tracking_Station_2], [distances_to_kerbin["Eve"][0]])

"""
need to enable to following business logic:

* range modifier: the listed power levels of all antenna are reduced (to 80% on medium and to 65% on hard) before calculating vessel power. This exclude the tracking station, which is unaffected.
* DSN Modifier: manually scale the power of the DSN network by this factor, which is 1.0 unless the player has manually reduced it in settings.
* Planet radii:
    * Moho: 250000
    * Eve: 700000
    * Gilly: 13000
    * Kerbin: 600000
    * Mun: 200000
    * Minmus: 60000
    * Duna: 320000
    * Ike: 130000
    * Dres: 150000
    * Jool: 600000
    * Laythe: 500000
    * Vall: 300000
    * Tylo: 700000
    * Bop: 65000
    * Pol: 45000
    * Eeloo: 210000
"""




    

0.26476064667237836